# 闭环试验：按代码格 1 → 6 顺序执行
每一步的标题说明目的、操作和通过条件。点击代码格，再按 Shift+Enter；等左侧 [*] 变回数字、输出停止后才读结果。不要 Run All。
如果已完成某项且内核没重启，从下一项继续；修改参数后必须重新执行参数格。
不要佩戴耳机。保持标定时箱体、麦克风、音响位置和接线，能立即切断音响电源。本轮无需讲话或播放手机音频，先保持环境安静。每次试验结束恢复官方配置并静音。
## 代码格 1：检查文件、加载函数（不发声）
操作：执行下面整格。看到 PACKAGE_HASHES_OK 和 CLOSED_TRIAL_READY 才继续到代码格2。报错就停止并保留完整输出。


In [ ]:
import hashlib, json
from pathlib import Path
manifest = json.loads(Path('manifest.json').read_text())
for name in ('audio_closed.bit', 'audio_closed.hwh', 'closed_trial.py', 'closed_tools.py', 'residual_tools.py'):
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == manifest['hashes']['board/audio_closed/'+name], name
print('PACKAGE_HASHES_OK')
%run closed_trial.py


## 代码格 2：设置本轮参数（不发声）
目的：固定两模式的音量、时长、夹具和模型，使下一步只改变开关。
操作：现场已按顶部说明准备好、同意采样保存在板卡后，把下面 speaker_confirmed=False 和 local_capture_confirmed=False 的 False 都改为 True。其余参数保持原样，执行整格（包括 filename 函数）。只改文字而没执行，参数不会生效。
- hp_volume=40：Codec音量，与标定一致，不要调整。
- gain_db=-36.0：PL降低音量，首轮保持此值。
- seconds=0.5：每次半秒，首轮保持此值。
执行后通常没有输出，且没有红色报错，即可继续代码格3。此前“0 dB仿真”是电脑测试条件，不要求你把实物增益改成0。


In [ ]:
from datetime import datetime
params = dict(fixture_id='box_v1_20261016',
              model_path='../audio_calibrate/candidate_box_v1_256.json',
              gain_db=-36.0, seconds=0.5, hp_volume=40,
              speaker_confirmed=False, local_capture_confirmed=False)
def filename(label):
    return label+'_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.npz'


## 代码格 3：零模型＋旁路（会运行半秒闭环）
目的：检查新通路能否正确传递麦克风数据、按时结束。零模型表示预测系数全部为0。
操作：执行下面整格一次，保持环境安静，等待完整输出；听不到很低的声音不算失败。
通过条件：出现 STOCK_BASE_RESTORED_AND_CODEC_MUTED；verification.bit_exact 为 true；prediction_saturations、residual_saturations、gain_saturations 都为0；结束后现场安静。全部满足才执行代码格4。否则停止，发回完整输出；出现尖叫立即断音响电源。


In [ ]:
zero_bypass = closed_trial(filename('zero_bypass'), bypass=True, zero_coeff=True, **params)
print(json.dumps(zero_bypass, indent=2))

## 代码格 4：零模型＋抵消开启（会运行半秒闭环）
目的：系数为0时，启用相减仍应保持算术正确；验证另一个分支。
操作：不改参数，不重复代码格3，执行下面整格一次。通过条件与代码格3相同。两次麦克风输入会变化，不要求RMS或录音文件完全相同。
全部满足可继续代码格5；否则停在这里，发回输出。


In [ ]:
zero_cancel = closed_trial(filename('zero_cancel'), bypass=False, zero_coeff=True, **params)
print(json.dumps(zero_cancel, indent=2))

## 代码格 5：真实模型＋旁路（会运行半秒闭环）
目的：装入标定系数，但暂不从麦克风减去预测，取得对照记录。
操作：零模型两项通过后，参数仍保持 -36 dB、0.5秒、hp_volume=40，执行下面整格一次。不要启动手机声源或移动箱体。
通过条件与代码格3相同。通过后继续代码格6；有错误或饱和则停止。


In [ ]:
bypass_result = closed_trial(filename('model_bypass'), bypass=True, **params)
print(json.dumps(bypass_result, indent=2))

## 代码格 6：真实模型＋抵消开启（会运行半秒闭环）
目的：实际使用预测相减，再通过相同增益送到DAC。
操作：不改参数，执行下面整格一次。通过条件仍与代码格3相同。
完成后把代码格4、5、6的完整文字输出及是否听到声音、结束是否安静发回；已发过的格3不用重复发送。本轮到这里结束，不自行升高增益或延长时长。


In [ ]:
cancel_result = closed_trial(filename('model_cancel'), bypass=False, **params)
print(json.dumps(cancel_result, indent=2))

## 本轮完成后怎么理解结果
四项核对的是板上闭环执行、算术和结束恢复；它们不代表ASG已经提高。后续再单独安排较长时长及同增益对照。
每项本地保存NPZ，不上传原始录音；文件名中的时间来自板卡时钟，时钟不准不影响本轮按帧数计时。试验保存末尾16384帧，不能据此宣称全程无啸叫。
如果内核重启导致 closed_trial/params/filename 未定义，重新执行格1、格2，再继续未完成项；不用重跑已完成试验。
